# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">RAG Basics</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know the basics of Retrieval-Augmented Generation (RAG), <br>
        ... can implement a RAG chain from scratch.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Nguyen (2024): Code a simple RAG from scratch](https://huggingface.co/blog/ngxson/make-your-own-rag)

In [ ]:
%pip install -q pymupdf4llm pymupdf_layout langchain-openai python-dotenv

## Where We Are

* **5.01:** the whole document goes into the prompt. Fine for a short document, too expensive for a long one.
* **Now:** send only the pieces that matter for the question. We build this from scratch, step by step.

## Retrieval-Augmented Generation

**In short: first find the relevant pieces, then let the LLM answer from them.**

![RAG](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/RAG.png)

(Image adapted from [Kaltenpoth and Müller (2024)](https://energy.acm.org/eir/dont-touch-the-power-line-a-proof-of-concept-for-aligned-llm-based-assistance-systems-to-support-the-maintenance-in-the-electricity-distribution-system/))

* The **retriever** (usually an embedding model) finds the k document pieces that are most similar to the question.
* These pieces are the **context**.
* The **generator**, an LLM, answers the question from this context.

RAG was introduced by [Lewis et al. (2020)](https://proceedings.neurips.cc/paper/2020/hash/6b493230205f780e1bc26945df7481e5-Abstract.html).

## Setup

In [ ]:
import os
import re
import urllib.request
import pymupdf4llm
from tqdm.notebook import tqdm
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Connect to the AI-Gateway

The same setup as in 4.03: your own computer, connected to the university network or the VPN, and your AI-Gateway key.

We use two models on the gateway:

* a **chat model** that writes the answers (the generator), and
* an **embedding model** that turns texts into vectors (the retriever).

**Good to know:** the gateway allows about 10 requests per minute per model. If a cell pauses, the client is waiting for the next minute and then continues by itself.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads the .env file: searches this folder first, then the folders above (the repo folder)
API_KEY = os.getenv("AI_GATEWAY_KEY")
AI_GATEWAY_BASE = os.getenv("AI_GATEWAY_BASE")  # address of the university's AI-Gateway
if not API_KEY or not AI_GATEWAY_BASE:
    raise ValueError("AI_GATEWAY_KEY or AI_GATEWAY_BASE missing. Copy .env.example to .env in the repo folder and paste your key there.")

GENERATOR_NAME = "gwdg.qwen3.8-27b-no-think"  # chat model: writes the answers
RETRIEVER_NAME = "gwdg.qwen3-embedding-4b"    # embedding model: turns texts into vectors

The key comes from the `.env` file in the repo folder (set up in 4.03). If the cell says that no key was found, copy `.env.example` to `.env` and paste your key behind `AI_GATEWAY_KEY=`.

In [ ]:
# downloads the documents, if they are not here yet
BASE_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in [
    "documents/Game_of_Thrones.pdf",
    "documents/How_I_Met_Your_Mother.pdf",
    "markdown_documents/Game_of_Thrones.md",
    "markdown_documents/How_I_Met_Your_Mother.md",
]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + path, path)

## Part 1: From PDF to Markdown

**In short: LLMs need text, so we convert the PDFs to Markdown.**

* We use two Wikipedia articles: *Game of Thrones* and *How I Met Your Mother*. With two topics, you can see that retrieval finds the right article and does not just get lucky.
* [PyMuPDF4LLM](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/) converts PDFs to Markdown for LLMs.

In [ ]:
documents_path = "documents"
markdown_documents_path = "markdown_documents"

In [ ]:
documents = sorted(os.listdir(documents_path))

for document in tqdm(documents):
    document_path = os.path.join(documents_path, document)
    md_file = pymupdf4llm.to_markdown(document_path)
    md_file_path = os.path.join(markdown_documents_path, document.replace(".pdf", ".md"))
    with open(md_file_path, "w", encoding="utf-8") as file:
        file.write(md_file)

## Part 2: Indexing

**In short: cut the documents into chunks, turn every chunk into a vector, and store the vectors.**

![Vector database](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/imgs/vectordb.png)

(Adapted from [Xie et al. (2023)](https://doi.org/10.1109/BigDIA60676.2023.10429609))

A vector database has two steps:

| Step | When | What happens |
|---|---|---|
| **Indexing** | once | the embedding model turns every chunk into a vector |
| **Querying** | for every question | a similarity search, for example with cosine similarity |

This is the indexing process that we build now:

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=1650&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=4b9e544a7a3ec168651558bce854eb60)

(Source: [Langchain Docs](https://docs.langchain.com/oss/python/langchain/rag#faiss))

### Loading Documents

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, file.read()])

In [ ]:
display(Markdown(md_files[0][1][:1000]))

In [ ]:
print(md_files[0][1][:1000])

The Markdown is full of Wikipedia links. They only distract the LLM, so we remove them.

In [ ]:
def remove_markdown_links(text: str) -> str:
    # 1. Remove inline links: [text](url)
    text = re.sub(r'\[([^\]]*)\]\([^)]+\)', r'\1', text)

    # 2. Remove reference-style links: [text][ref]
    text = re.sub(r'\[([^\]]*)\]\[[^\]]*\]', r'\1', text)

    # 3. Remove malformed/broken bracket groups like [[][1][]]
    text = re.sub(r'\[+\s*\]+', '', text)

    # 4. Remove reference definitions like: [1]: https://...
    text = re.sub(r'^\s*\[[^\]]+\]:.*$', '', text, flags=re.MULTILINE)

    return text

In [ ]:
display(Markdown(remove_markdown_links(md_files[0][1])[:1000]))

Now the text is much easier to read.

In [ ]:
markdown_documents = sorted(os.listdir(markdown_documents_path))

md_files = []

for markdown_document in markdown_documents:
    markdown_document_path = os.path.join(markdown_documents_path, markdown_document)
    with open(markdown_document_path, encoding="utf-8") as file:
        md_files.append([markdown_document, remove_markdown_links(file.read())])

### Chunking Texts

Whole documents are too long for one embedding and too long for a prompt. So we cut them into **chunks**.

We cut by characters: in English, about 4 characters are one token, so 2,000 characters are roughly 500 tokens. Neighbouring chunks share some text (the **overlap**), so that a sentence at the border is not torn apart ([Wang et al., 2024](https://doi.org/10.18653/v1/2024.emnlp-main.981)).

In [ ]:
for md_file in md_files:
    print("File name:", md_file[0], "| characters:", len(md_file[1]), "| about", len(md_file[1]) // 4, "tokens")

In [ ]:
OVERLAP = 200        # characters shared by two neighbouring chunks
CHUNK_LENGTH = 2000  # characters per chunk, roughly 500 tokens

md_files_chunked = []
for md_file in md_files:
    text = md_file[1]
    for start in range(0, len(text), CHUNK_LENGTH - OVERLAP):
        md_files_chunked.append([md_file[0], text[start:start + CHUNK_LENGTH]])

print(len(md_files_chunked), "chunks")

In [ ]:
print(md_files_chunked[0])
print(md_files_chunked[1])

### Loading the Embedding Model

The embedding model runs on the AI-Gateway. LangChain's `OpenAIEmbeddings` is the embedding counterpart of `ChatOpenAI` from 5.01: same `base_url`, same key.

In [ ]:
embedding_model = OpenAIEmbeddings(
    model=RETRIEVER_NAME,
    base_url=AI_GATEWAY_BASE,
    api_key=API_KEY,
    check_embedding_ctx_length=False,  # send plain text; the default would first split it with OpenAI's own tokenizer
)

In [ ]:
embeddings = embedding_model.embed_documents([md_files_chunked[0][1]])

In [ ]:
print(list(embeddings[0][:10]) + ["..."], len(embeddings[0]))

### The vector database

We use the easiest (and slowest) way to store the vectors: a plain Python list.

We send **all** chunks in **one** request, because the gateway allows only a few requests per minute. This cell still takes about a minute and uses about 150,000 tokens of your budget, so run it only once.

In [ ]:
all_embeddings = embedding_model.embed_documents([chunk for source, chunk in md_files_chunked])

VECTOR_DB = []
for (source, chunk), embedding in zip(md_files_chunked, all_embeddings):
    VECTOR_DB.append({"embeddings": embedding, "content": chunk, "metadata": {"source": source}})

print(len(VECTOR_DB), "chunks in the vector database")

In [ ]:
def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

## Part 3: Retrieval

**In short: embed the question, and return the chunks that are most similar to it.**

### Your turn

Complete `retrieve()`: compute the cosine similarity between the question and every chunk in `VECTOR_DB`, and add a `[chunk, similarity]` pair to `similarities` for each one.

In [ ]:
def retrieve(query, top_n=3):
    query_embedding = embedding_model.embed_query(query)
    # temporary list to store (chunk, similarity) pairs
    similarities = []
    # TODO: calculate cosine similarity between query and each chunk in the VECTOR_DB
    # Hint: each VECTOR_DB entry is a dictionary with keys "embeddings" (embeddings) and "content" (text chunk)



    # write your code here




    # sort by similarity in descending order, because higher similarity means more relevant chunks
    similarities.sort(key=lambda x: x[1], reverse=True)
    # finally, return the top N most relevant chunks
    return similarities[:top_n]

In [ ]:
retrieved_docs = retrieve("Who plays Jon Snow?")

for doc in retrieved_docs:
    print(doc)

## Part 4: Generation

**In short: the LLM answers the question from the retrieved chunks only.**

The generator is the chat model on the gateway, with `ChatOpenAI` from 5.01.

In [ ]:
llm = ChatOpenAI(model=GENERATOR_NAME, base_url=AI_GATEWAY_BASE, api_key=API_KEY, temperature=0)

### The generation prompt

In [ ]:
prompt = '''Use only the following context chunks to answer the question: {input_query}
Don't make up any new information. Here are the chunks:
{chunks}
Now answer the question: {input_query}'''

### Retrieve, then generate

Now let's bring everything together. `generate()` sends a list of messages to the generator and returns the text of the answer.

In [ ]:
def generate(messages):
    return llm.invoke(messages).content

#### Step 1: Retrieve

In [ ]:
input_query = "Who plays Jon Snow?"
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
    print(f' - (similarity: {similarity:.2f}) {chunk}')

#### Step 2: Generate

In [ ]:
chunks = '\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])
question = prompt.format(input_query=input_query, chunks=chunks)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question}
]
response = generate(messages)

In [ ]:
print(response)

## Part 5: The RAG Chain

**In short: retrieval and generation in one function.**

### Your turn

Complete `generate_rag_response()`: retrieve the chunks, print them, fill the prompt, generate the answer, and return it. You find every piece in the cells above.

In [ ]:
def generate_rag_response(input_query):
    # TODO: retrieve relevant chunks from the VECTOR_DB
    # TODO: print chunks
    # TODO: create a prompt using the retrieved chunks and the input query
    # TODO: generate a response using the RAG model
    # TODO: return the generated response


    # write your code here



    pass

### Try it

Ask your own question about one of the two series.

In [ ]:
input_query = input('Ask me a question: ')
print(generate_rag_response(input_query))

<a href="https://imgflip.com/i/9fsgxc"><img src="https://i.imgflip.com/9fsgxc.jpg" title="made at imgflip.com"/></a><div><a href="https://imgflip.com/memegenerator">from Imgflip Meme Generator</a></div>

## What You Learned

* **RAG** = first retrieve the relevant chunks, then generate the answer from them.
* **Indexing** (once): PDF → Markdown → chunks → embeddings → vector database.
* **Querying** (for every question): embed the question and find the most similar chunks with cosine similarity.
* The generator sees only the retrieved chunks, not the whole document. That saves tokens.

Next: **5.03 RAG in 10 Lines of Code**. The same with LangChain.